# **Python**

## Age Weighted Missing Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def _save_confusion_table_png(var_name, counts, accuracy, precision, recall, f1, spec, out_path):
    """
    Gera uma imagem PNG com a matriz de confusão (2x2) formatada em tabela.
    counts: dict com TN, FP, FN, TP (int)
    """
    # Matriz no padrão:
    # linhas = Verdadeiro (0, 1)
    # colunas = Predito   (0, 1)
    matrix = np.array([
        [counts["TN"], counts["FP"]],
        [counts["FN"], counts["TP"]],
    ], dtype=int)

    fig, ax = plt.subplots(figsize=(4, 3.2), dpi=200)  # gráfico único (sem subplots múltiplos)
    ax.axis('off')

    # Cabeçalho
    title = f"Matriz de Confusão — {var_name}\nAccuracy: {accuracy:.3f}\nPrecision: {precision:.3f}\nRecall: {recall:.3f}\nF1: {f1:.3f}\nSpecificity: {spec:.3f}"
    ax.text(0.5, 1.05, title, ha='center', va='bottom', fontsize=10, transform=ax.transAxes)

    # Tabela
    col_labels = ["Pred 0", "Pred 1"]
    row_labels = ["True 0", "True 1"]
    table = ax.table(cellText=matrix.astype(str),
                     rowLabels=row_labels,
                     colLabels=col_labels,
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.4)

    # Destaque leve no header
    for (row, col), cell in table.get_celld().items():
        if row == 0 or col == -1:
            cell.set_text_props(weight='bold')

    fig.tight_layout(pad=0.6)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


def avaliar_imputacao(
    caminho_original="sample_signs_comorb_2019.xlsx",
    caminho_com_missing="age_weighted_missing_data_sample_signs_comorb_2019.xlsx",
    caminho_imputada="pooled_age.xlsx",
    cutoff=0.5,
    sheet_name=0,
    pasta_imgs="confusion_matrices"
):
    # 1) Carregar dados
    df_orig = pd.read_excel(caminho_original, sheet_name=sheet_name)
    df_miss = pd.read_excel(caminho_com_missing, sheet_name=sheet_name)
    df_imp  = pd.read_excel(caminho_imputada, sheet_name=sheet_name)

    # 2) Alinhar colunas (caso a ordem difira)
    if not (list(df_orig.columns) == list(df_miss.columns) == list(df_imp.columns)):
        cols = list(set(df_orig.columns) & set(df_miss.columns) & set(df_imp.columns))
        df_orig = df_orig[cols].copy()
        df_miss = df_miss[cols].copy()
        df_imp  = df_imp[cols].copy()

    # 3) Verificação de linhas
    if len(df_orig) != len(df_miss) or len(df_orig) != len(df_imp):
        raise ValueError("As três bases precisam ter o MESMO número de linhas (mesmo índice/ordem).")

    # 4) Onde havia missing sintético
    miss_mask = df_miss.isna()
    cols_avaliar = [c for c in df_miss.columns if miss_mask[c].any()]

    # 5) Saídas
    linhas_confusao = []

    pasta_imgs = Path(pasta_imgs)

    # 6) Loop por variável
    for col in cols_avaliar:
        idx = miss_mask[col].values
        if idx.sum() == 0:
            continue

        # Verdade (original) e predito (imputado) nas posições com missing sintético
        y_true = pd.to_numeric(df_orig.loc[idx, col], errors="coerce").astype(float).values
        y_pred = pd.to_numeric(df_imp.loc[idx, col],  errors="coerce").astype(float).values

        # --------- Matriz de confusão (cutoff p/ binarizar imputação)
        y_pred_bin = (y_pred >= cutoff).astype(int)
        # robusto se y_true for 0.0/1.0
        y_true_bin = (y_true >= 0.5).astype(int)

        tp = int(((y_pred_bin == 1) & (y_true_bin == 1)).sum())
        tn = int(((y_pred_bin == 0) & (y_true_bin == 0)).sum())
        fp = int(((y_pred_bin == 1) & (y_true_bin == 0)).sum())
        fn = int(((y_pred_bin == 0) & (y_true_bin == 1)).sum())

        total = tp + tn + fp + fn

        # Métricas
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        spec = (tn/(tn+fp))

        linhas_confusao.append({
            "variavel": col,
            "cutoff": cutoff,
            "n_avaliado": total,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "specificity": spec
        })

        # --------- Salvar PNG
        out_png = pasta_imgs / f"confusion_matrix_{col}.png"
        _save_confusion_table_png(
            var_name=col,
            counts={"TN": tn, "FP": fp, "FN": fn, "TP": tp},
            accuracy = accuracy_score(y_true, y_pred),
            precision = precision_score(y_true, y_pred, zero_division=0),
            recall = recall_score(y_true, y_pred, zero_division=0),
            f1 = f1_score(y_true, y_pred, zero_division=0),
            spec = (tn/(tn+fp)),
            out_path=out_png
            )

    # 7) DataFrames e CSVs
    df_confusao = pd.DataFrame(linhas_confusao).sort_values("variavel").reset_index(drop=True)

    df_confusao.to_csv("matrizes_confusao_por_variavel.csv", index=False)

    return df_confusao


if __name__ == "__main__":
    confusoes = avaliar_imputacao(
        caminho_original="sample_signs_comorb_2019.xlsx",
        caminho_com_missing="age_weighted_missing_data_sample_signs_comorb_2019.xlsx",
        caminho_imputada="pooled_age.xlsx",
        cutoff=0.5,
        sheet_name=0,
        pasta_imgs="confusion_matrices"
    )

    print('VERTEX MICE IMPUTER PERFORMANCE TO SINAN AGE WEIGHTED MISSING DATA')

    print("\n=== Matriz de confusão por variável (primeiras linhas) ===")
    print(confusoes.head(20).to_string(index=False))

VERTEX MICE IMPUTER PERFORMANCE TO SINAN AGE WEIGHTED MISSING DATA

=== Matriz de confusão por variável (primeiras linhas) ===
  variavel  cutoff  n_avaliado  TN  FP  FN  TP  accuracy  precision  recall       f1  specificity
ACIDO_PEPT     0.5         200 198   0   2   0     0.990       0.00     0.0 0.000000     1.000000
AUTO_IMUNE     0.5         200 199   0   0   1     1.000       1.00     1.0 1.000000     1.000000
  DIABETES     0.5         200 196   0   4   0     0.980       0.00     0.0 0.000000     1.000000
 HEMATOLOG     0.5         200 199   0   1   0     0.995       0.00     0.0 0.000000     1.000000
 HEPATOPAT     0.5         200 198   0   2   0     0.990       0.00     0.0 0.000000     1.000000
HIPERTENSA     0.5         200 184   1  12   3     0.935       0.75     0.2 0.315789     0.994595
     RENAL     0.5         200 199   0   1   0     0.995       0.00     0.0 0.000000     1.000000


# **R**

## Pooling

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mode

# Lista dos arquivos imputados
files = [
    "imputed_weighted_1.csv",
    "imputed_weighted_2.csv",
    "imputed_weighted_3.csv",
    "imputed_weighted_4.csv",
    "imputed_weighted_5.csv"
]

# Ler todos os arquivos
dfs = [pd.read_csv(f) for f in files]

# Remover possíveis colunas de índice (tipo unnamed)
for i in range(len(dfs)):
    dfs[i] = dfs[i].loc[:, ~dfs[i].columns.str.contains('^Unnamed')]

# Garantir que todas as colunas estejam na mesma ordem
columns = dfs[0].columns
dfs = [df[columns] for df in dfs]

# Separar tipos de variáveis
num_cols = dfs[0].select_dtypes(include=[np.number]).columns
cat_cols = dfs[0].select_dtypes(exclude=[np.number]).columns

# Fazer pooling (média para numéricas, moda para categóricas)
pooled_df = pd.DataFrame()

# Média das numéricas
for col in num_cols:
    pooled_df[col] = np.mean([df[col] for df in dfs], axis=0)

# Moda das categóricas
for col in cat_cols:
    # Stack the dataframes for the current column and find the mode for each row
    stacked_col = pd.concat([df[col] for df in dfs], axis=1)
    pooled_df[col] = stacked_col.mode(axis=1)[0] # mode(axis=1) calculates the mode for each row

# Salvar resultado final em Excel
pooled_df.to_excel("imputed_weighted_r_converged.xlsx", index=False)

print("Pooling concluído e salvo")

Pooling concluído e salvo


## Age Weighted Missing Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def _save_confusion_table_png(var_name, counts, accuracy, precision, recall, f1, spec, out_path):
    """
    Gera uma imagem PNG com a matriz de confusão (2x2) formatada em tabela.
    counts: dict com TN, FP, FN, TP (int)
    """
    # Matriz no padrão:
    # linhas = Verdadeiro (0, 1)
    # colunas = Predito   (0, 1)
    matrix = np.array([
        [counts["TN"], counts["FP"]],
        [counts["FN"], counts["TP"]],
    ], dtype=int)

    fig, ax = plt.subplots(figsize=(4, 3.2), dpi=200)  # gráfico único (sem subplots múltiplos)
    ax.axis('off')

    # Cabeçalho
    title = f"Matriz de Confusão — {var_name}\nAccuracy: {accuracy:.3f}\nPrecision: {precision:.3f}\nRecall: {recall:.3f}\nF1: {f1:.3f}\nSpecificity: {spec:.3f}"
    ax.text(0.5, 1.05, title, ha='center', va='bottom', fontsize=10, transform=ax.transAxes)

    # Tabela
    col_labels = ["Pred 0", "Pred 1"]
    row_labels = ["True 0", "True 1"]
    table = ax.table(cellText=matrix.astype(str),
                     rowLabels=row_labels,
                     colLabels=col_labels,
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.4)

    # Destaque leve no header
    for (row, col), cell in table.get_celld().items():
        if row == 0 or col == -1:
            cell.set_text_props(weight='bold')

    fig.tight_layout(pad=0.6)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


def avaliar_imputacao(
    caminho_original="sample_signs_comorb_2019.xlsx",
    caminho_com_missing="age_weighted_missing_data_sample_signs_comorb_2019.xlsx",
    caminho_imputada="imputed_weighted_r_converged.xlsx",
    cutoff=0.5,
    sheet_name=0,
    pasta_imgs="confusion_matrices"
):
    # 1) Carregar dados
    df_orig = pd.read_excel(caminho_original, sheet_name=sheet_name)
    df_miss = pd.read_excel(caminho_com_missing, sheet_name=sheet_name)
    df_imp  = pd.read_excel(caminho_imputada, sheet_name=sheet_name)

    # 2) Alinhar colunas (caso a ordem difira)
    if not (list(df_orig.columns) == list(df_miss.columns) == list(df_imp.columns)):
        cols = list(set(df_orig.columns) & set(df_miss.columns) & set(df_imp.columns))
        df_orig = df_orig[cols].copy()
        df_miss = df_miss[cols].copy()
        df_imp  = df_imp[cols].copy()

    # 3) Verificação de linhas
    if len(df_orig) != len(df_miss) or len(df_orig) != len(df_imp):
        raise ValueError("As três bases precisam ter o MESMO número de linhas (mesmo índice/ordem).")

    # 4) Onde havia missing sintético
    miss_mask = df_miss.isna()
    cols_avaliar = [c for c in df_miss.columns if miss_mask[c].any()]

    # 5) Saídas
    linhas_confusao = []

    pasta_imgs = Path(pasta_imgs)

    # 6) Loop por variável
    for col in cols_avaliar:
        idx = miss_mask[col].values
        if idx.sum() == 0:
            continue

        # Verdade (original) e predito (imputado) nas posições com missing sintético
        y_true = pd.to_numeric(df_orig.loc[idx, col], errors="coerce").astype(float).values
        y_pred = pd.to_numeric(df_imp.loc[idx, col],  errors="coerce").astype(float).values

        # --------- Matriz de confusão (cutoff p/ binarizar imputação)
        y_pred_bin = (y_pred >= cutoff).astype(int)
        # robusto se y_true for 0.0/1.0
        y_true_bin = (y_true >= 0.5).astype(int)

        tp = int(((y_pred_bin == 1) & (y_true_bin == 1)).sum())
        tn = int(((y_pred_bin == 0) & (y_true_bin == 0)).sum())
        fp = int(((y_pred_bin == 1) & (y_true_bin == 0)).sum())
        fn = int(((y_pred_bin == 0) & (y_true_bin == 1)).sum())

        total = tp + tn + fp + fn

        # Métricas
        accuracy = accuracy_score(y_true_bin, y_pred_bin)
        precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        spec = (tn/(tn+fp))

        linhas_confusao.append({
            "variavel": col,
            "cutoff": cutoff,
            "n_avaliado": total,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "specificity": spec
        })

        # --------- Salvar PNG
        out_png = pasta_imgs / f"confusion_matrix_{col}.png"
        _save_confusion_table_png(
            var_name=col,
            counts={"TN": tn, "FP": fp, "FN": fn, "TP": tp},
            accuracy = accuracy_score(y_true_bin, y_pred_bin),
            precision = precision_score(y_true_bin, y_pred_bin, zero_division=0),
            recall = recall_score(y_true_bin, y_pred_bin, zero_division=0),
            f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0),
            spec = (tn/(tn+fp)),
            out_path=out_png
            )

    # 7) DataFrames e CSVs
    df_confusao = pd.DataFrame(linhas_confusao).sort_values("variavel").reset_index(drop=True)

    df_confusao.to_csv("matrizes_confusao_por_variavel.csv", index=False)

    return df_confusao


if __name__ == "__main__":
    confusoes = avaliar_imputacao(
        caminho_original="sample_signs_comorb_2019.xlsx",
        caminho_com_missing="age_weighted_missing_data_sample_signs_comorb_2019.xlsx",
        caminho_imputada="imputed_weighted_r_converged.xlsx",
        cutoff=0.5,
        sheet_name=0,
        pasta_imgs="confusion_matrices"
    )

    print('R MICE CART PERFORMANCE TO SINAN AGE WEIGHTED MISSING DATA')

    print("\n=== Matriz de confusão por variável (primeiras linhas) ===")
    print(confusoes.head(20).to_string(index=False))

R MICE CART PERFORMANCE TO SINAN AGE WEIGHTED MISSING DATA

=== Matriz de confusão por variável (primeiras linhas) ===
  variavel  cutoff  n_avaliado  TN  FP  FN  TP  accuracy  precision   recall      f1  specificity
ACIDO_PEPT     0.5         200 198   0   2   0     0.990   0.000000 0.000000 0.00000     1.000000
AUTO_IMUNE     0.5         200 199   0   1   0     0.995   0.000000 0.000000 0.00000     1.000000
  DIABETES     0.5         200 192   4   4   0     0.960   0.000000 0.000000 0.00000     0.979592
 HEMATOLOG     0.5         200 199   0   1   0     0.995   0.000000 0.000000 0.00000     1.000000
 HEPATOPAT     0.5         200 198   0   2   0     0.990   0.000000 0.000000 0.00000     1.000000
HIPERTENSA     0.5         200 178   7  10   5     0.915   0.416667 0.333333 0.37037     0.962162
     RENAL     0.5         200 199   0   1   0     0.995   0.000000 0.000000 0.00000     1.000000
